# Problem Reductions and SAT Solvers

When we have a problem reduction, $A\leq B$, then we have a way to transform $A$-questions into 
$B$-questions.
So if we can find answers about questions from $B$ then we can use those answers to solve questions from $A$.

Here, $A$ is Soduku and $B$ is $\mathcal{SAT}$.
We will develop Racket routines that give a function witnessing the reduction between problems, that input Soduku problems and that output $\mathcal{SAT}$ problems.
Then to solve the Soduku 
we feed this propositional logic formula to a $\mathcal{SAT}$ solver program called MiniSat, and finally we interpret the answer.

## Conjunctive Normal form

A propositional logic expression in the form $(A\vee B)\wedge(\neg B \vee C)$
or $(P\vee Q\vee \neg R)\wedge S \wedge (\neg Q \vee R)$ is in 
*conjunctive normal form (CNF)*.
That is, the expression is an AND of OR's: it is the conjunction of clauses, where each clause is the
disjunction of literals (a literal is either a singel atom, such as $X$ or the negation of an atom, such as $\neg X$).

This form is maximally expressive, in that for any Boolean function there is a CNF expression that
gives it.
Here is an example to show the general pattern.
The truth table on the left below describes a Boolean function, $f$
(on the right is its negation, $\neg f$).
We want a propositional logic expression that generates this table, 
and that is in CNF.
\begin{equation*}
  \begin{array}{ccc|c}
    P &Q &R &f \\
    \hline
    F &F &F &F  \\
    F &F &T &F  \\
    F &T &F &T  \\
    F &T &T &F  \\
    T &F &F &F  \\
    T &F &T &T  \\
    T &T &F &F  \\
    T &T &T &T  \\
  \end{array}
  \qquad
  \begin{array}{ccc|c}
    P &Q &R &\neg f \\
    \hline
    F &F &F &T  \\
    F &F &T &T  \\
    F &T &F &F  \\
    F &T &T &T  \\
    T &F &F &T  \\
    T &F &T &F  \\
    T &T &F &T  \\
    T &T &T &F  \\
  \end{array}
\end{equation*}
For the table on the right, we know that we get $T$ if and only if one of these holds: the first line,
the second, the third, the fourth, the fifth line, or the seventh line.
Thus, $\neg f$ is true if and only if the expression
$(\neg P\wedge \neg Q\wedge \neg R)
\vee (\neg P\wedge \neg Q\wedge R)
\vee (\neg P\wedge Q\wedge R)
\vee (P\wedge \neg Q\wedge \neg R)
\vee (P\wedge Q\wedge \neg R)$ evaluates to true.
This expression is not in CNF
(it is in a form called Disjunctive Normal form).
But our real interest is that $f$ is true if and only if the negation of this expression is true.
\begin{equation*}
f\equiv 
\neg \bigl[\,(\neg P\wedge \neg Q\wedge \neg R)
\vee (\neg P\wedge \neg Q\wedge R)
\vee (\neg P\wedge Q\wedge R)
\vee (P\wedge \neg Q\wedge \neg R)
\vee (P\wedge Q\wedge \neg R)\,\bigr]
\end{equation*}
Apply DeMorgan's Law to distribute the outer negation over the $\vee$'s. 
\begin{equation*}
f\equiv
\neg(\neg P\wedge \neg Q\wedge \neg R)
\wedge \neg(\neg P\wedge \neg Q\wedge R)
\wedge \neg(\neg P\wedge Q\wedge R)
\wedge \neg(P\wedge \neg Q\wedge \neg R)
\wedge \neg(P\wedge Q\wedge \neg R)
\end{equation*}
Finish by applying DeMorgan's Laws again, this time to each clause. 
\begin{equation*}
f\equiv
(P\vee Q\vee R)
\wedge (P\vee Q\vee\neg R)
\wedge (P\vee\neg Q\vee\neg R)
\wedge (\neg P\vee Q\vee R)
\wedge (\neg P\vee\neg Q\vee R)
\end{equation*}
That expression is in CNF.

#### Exercise

*Give a CNF form expression for this truth table.*
\begin{equation*}
  \begin{array}{ccc|c}
    P &Q &R &g \\
    \hline
    F &F &F &T  \\
    F &F &T &F  \\
    F &T &F &F  \\
    F &T &T &T  \\
    T &F &F &T  \\
    T &F &T &F  \\
    T &T &F &T  \\
    T &T &T &T  \\
  \end{array}
\end{equation*}

## Reducing Soduku to $\mathcal{SAT}$

Recall that 
the input to Soduku is a 9-by-9 array with some of the cells already filled in.

![Initial Soduku board](img/soduku0.png)

A solved problem fills in the all of the blanks, subject to some restrictions.
Each of the rows $R_1, \ldots\, R_9$ must contain each of the numbers in the set $\{1,\ldots\,9\}$.
Likewise, each of the columns $C_1, \ldots\, C_9$ must contain each of the numbers in $\{1,\ldots\,9\}$.
Further, there are nine subsquares that have to satisfy the same criteria, that the subsquare contains
each of the numbers in $\{1,\ldots\,9\}$.
The first subsquare consists of the nine entries in the upper left
$S_1=\{x_{1,1},x_{1,2},x_{1,3},\ldots\,x_{3,3}$.
The second subsquare $S_2$ contains the nine entries in the upper middle, etc., and the
last subsquare contains the nine entries in the lower right
$S_9=\{x_{7,7},x_{7,8},x_{7,9},\ldots\,x_{9,9}\}$.

*Remark.*
This problem naturally generalizes.
Instead of eighty one variables $x_{1,1},\ldots\, x_{9,9}$, we can have an arbitrary number, $x_1, \ldots\, x_n$.
Instead of having the variables take on the values $1,\ldots\, 9$, we can look to give them
some value in a range $\{ 1,\ldots\, k\}$ (instead of `values', some authors refer to these as `colors').
And instead of geometrically-driven rows, columns, etc.,
an instance of the problem will just have arbitrary size-$k$ sets
$S_i\subseteq\{x_1,\ldots\,x_n\}$.
With that, we can show that the Soduku problem is $\mathcal{NP}$-complete.
However, having said that, we will ignore it.
We will just show how to go from a given traditional 9-by-9 Soduku board instance to an
instance of $\mathcal{SAT}$.

We will produce a function that inputs Soduku problems, the ordinary 9-by-9 kind, and outputs CNF expressions that
are satisfiable if and only if the original problem is solvable.
We will implement this function in Racket.

## The reduction

The $\mathcal{SAT}$ problem that we produce will be a CNF expression with two kinds of clauses.
One kind describe the general rules of Soduku, and the other is specific to the particular board.
It is as though we bought a Soduku puzzle book and we opened first to the Preface, describing the rules
(a row contains each number 1-9, as does a column, and as does each of the
nine squares), and then also opened to the page containing the specific partial board shown above.
We will first cover the general rule clauses, and then the specific board.

We will have many variables.
For instance, for the row 1, column 1 entry, we will have a Boolean variable $x_{1,1,1}$, 
and a variable $x_{1,1,2}$, etc.,
up to a variable $x_{1,1,9}$.
Only one of them will be $T$, and the rest will be $F$.
The variable $x_{1,1,v}$ is $T$ if in our solution the number in the row 1 and column 1 entry is $v$.
Otherwise it is $F$.


